# 11 — Can Quantum Feature Maps Model Nonlinear Dynamics in Arbitrage-Free Implied-Volatility Surfaces?

**Project:** SSVI Volatility Surface Parameters as Predictors — Quantum Kernel Extension
**Politecnico di Milano — Insurance & Econometrics, A.Y. 2024/25**

---

> **Framing:** This notebook tests quantum feature maps as an alternative nonlinear representation
> learning tool for volatility-surface dynamics. The goal is **not** to prove quantum advantage,
> but to evaluate whether quantum kernels provide incremental predictive information over a naive
> baseline and a conservative Gradient Boosting model.

**Research question:** *Can ZZFeatureMap quantum kernels capture nonlinear dependencies in SSVI
parameter changes that are missed by standard linear/tree-based models?*

**Three-model horse race:**
1. **Naive baseline** — δ̂ = 0 (tomorrow's parameter = today's)
2. **Gradient Boosting** — conservative settings, early stopping, no aggressive tuning
3. **Quantum Kernel SVR** — ZZFeatureMap + FidelityQuantumKernel + SVR(kernel="precomputed")

**Anti-leakage rules (strictly enforced):**
- All features use only information available at time *t*
- Scaler, PCA, feature selection fit **only on train**
- Temporal 80/20 split — no random shuffle
- Targets constructed with `shift(-1)` then `dropna`


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR

# ── Qiskit availability check ─────────────────────────────────────────────────
QISKIT_AVAILABLE   = False
QISKIT_KERNEL_API  = None

try:
    from qiskit.circuit.library import ZZFeatureMap
    try:
        from qiskit_machine_learning.kernels import FidelityQuantumKernel
        QISKIT_KERNEL_API = "fidelity"
    except ImportError:
        try:
            from qiskit_machine_learning.kernels import QuantumKernel
            QISKIT_KERNEL_API = "legacy"
        except ImportError:
            QISKIT_KERNEL_API = None
    if QISKIT_KERNEL_API is not None:
        QISKIT_AVAILABLE = True
        print(f"Qiskit Machine Learning: available  (API={QISKIT_KERNEL_API})")
    else:
        print("ZZFeatureMap found but no kernel class — check qiskit-machine-learning version")
except ImportError:
    print("Qiskit not available. To install:")
    print("  pip install qiskit qiskit-machine-learning")
    print("Quantum section will be SKIPPED — Naive + GradientBoosting will still run.")

OUTPUT_DIR = Path("output")
PLOT_DIR   = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output  : {OUTPUT_DIR.resolve()}")
print(f"Plots   : {PLOT_DIR.resolve()}")


## Data Loading

SSVI parameters from `ssvi_forecasting_dataset.csv` (NB06 output). VIX and VVIX merged from the market data cache (FRED). No proxies — if VVIX is absent the section runs without it.


In [ ]:
# ── SSVI parameters ───────────────────────────────────────────────────────────
ssvi = pd.read_csv(OUTPUT_DIR / "ssvi_forecasting_dataset.csv")
BASE_DATE    = pd.Timestamp("2010-01-04")
ssvi["date"] = BASE_DATE + pd.to_timedelta(ssvi["time_elapsed"], unit="D")
ssvi         = ssvi.sort_values("date").reset_index(drop=True)

# ── Market data: VIX, VVIX (from FRED cache) ──────────────────────────────────
VIX_AVAILABLE  = False
VVIX_AVAILABLE = False
mkt_path = OUTPUT_DIR / "_cache_mh_fred.csv"

if mkt_path.exists():
    mkt = pd.read_csv(mkt_path, index_col=0, parse_dates=True)
    mkt.index = pd.to_datetime(mkt.index).normalize()
    mkt_r = mkt.reset_index().rename(columns={mkt.index.name or "index": "date"})
    mkt_r["date"] = pd.to_datetime(mkt_r["date"])
    df = pd.merge_asof(ssvi.sort_values("date"), mkt_r.sort_values("date"),
                       on="date", direction="nearest", tolerance=pd.Timedelta("3D"))
    VIX_AVAILABLE  = "VIX"  in df.columns and df["VIX"].notna().sum() > 100
    VVIX_AVAILABLE = "VVIX" in df.columns and df["VVIX"].notna().sum() > 100
    print(f"Market merge: VIX={VIX_AVAILABLE}  VVIX={VVIX_AVAILABLE}")
else:
    df = ssvi.copy()
    print("No market data cache — using SSVI parameters only")

print(f"Dataset shape: {df.shape}")
print(f"Date range  : {df['date'].min().date()} — {df['date'].max().date()}")
print(f"Samples     : train={( df['sample']=='train').sum()}  test={(df['sample']=='test').sum()}")


## Feature Engineering & Target Construction

Targets are one-step-ahead SSVI parameter deltas: `Δp_{t+1} = p_{t+1} − p_t`. No leakage: `shift(-1)` on the sorted series, then `dropna`.


In [ ]:
SSVI_PARAMS = ["alpha", "beta", "rho", "eta", "gamma"]
params_in_data = [p for p in SSVI_PARAMS if p in df.columns]
print(f"SSVI parameters: {params_in_data}")

# ── Targets: next-period delta (no leakage) ───────────────────────────────────
TARGETS = {}
for p in params_in_data:
    # delta today  = param_t  - param_{t-1}  (already as p - p_lag1)
    df[f"d_{p}"] = df[p] - df.get(f"{p}_lag1", df[p].shift(1))
    # target = param_{t+1} - param_t = delta tomorrow
    tgt_col = f"d_{p}_next"
    df[tgt_col] = df[p].shift(-1) - df[p]
    TARGETS[p] = tgt_col

# ── Log RV features ───────────────────────────────────────────────────────────
if "RV5" in df.columns:
    df["lrv5"]  = np.log(df["RV5"].clip(1e-10))
if "RV20" in df.columns:
    df["lrv20"] = np.log(df["RV20"].clip(1e-10))

# ── VIX / VVIX log features ───────────────────────────────────────────────────
if VIX_AVAILABLE:
    df["log_VIX"]  = np.log(df["VIX"].clip(1e-3))
if VVIX_AVAILABLE:
    df["log_VVIX"] = np.log(df["VVIX"].clip(1e-3))

# ── Base feature set (all backward-looking at time t) ────────────────────────
BASE_FEATURES = []

# Current SSVI levels
BASE_FEATURES += params_in_data

# Computed deltas (today's change)
BASE_FEATURES += [f"d_{p}" for p in params_in_data]

# Lagged deltas (already in dataset: p_lag1, p_lag2)
for p in params_in_data:
    lag1_col = f"{p}_lag1"
    lag2_col = f"{p}_lag2"
    if lag1_col in df.columns and lag2_col in df.columns:
        df[f"d_{p}_lag1"] = df[lag1_col] - df[lag2_col]
        BASE_FEATURES.append(f"d_{p}_lag1")

# RV and return features
for col in ["lrv5", "lrv20", "ret_5d", "ret_20d", "log_return"]:
    if col in df.columns:
        BASE_FEATURES.append(col)

# Surface shape features (already engineered in NB06)
for col in ["abs_rho", "eta_gamma_interaction", "skew_stress"]:
    if col in df.columns:
        BASE_FEATURES.append(col)

# Market data
if VIX_AVAILABLE:
    BASE_FEATURES.append("log_VIX")
if VVIX_AVAILABLE:
    BASE_FEATURES.append("log_VVIX")

# Deduplicate preserving order
seen = set(); BASE_FEATURES = [f for f in BASE_FEATURES if not (f in seen or seen.add(f))]

print(f"Feature count : {len(BASE_FEATURES)}")
print(f"Features      : {BASE_FEATURES}")
print(f"Targets       : {list(TARGETS.values())}")

# ── Drop NaN ──────────────────────────────────────────────────────────────────
required = BASE_FEATURES + list(TARGETS.values())
df_clean = df.dropna(subset=required).reset_index(drop=True)
print(f"Clean dataset : {df_clean.shape}  (dropped {len(df) - len(df_clean)} rows with NaN)")


In [ ]:
# ── Temporal 80/20 holdout split ─────────────────────────────────────────────
N      = len(df_clean)
SPLIT  = int(N * 0.80)
df_train = df_clean.iloc[:SPLIT].copy().reset_index(drop=True)
df_test  = df_clean.iloc[SPLIT:].copy().reset_index(drop=True)

print(f"Train: {len(df_train):5d} obs  "
      f"({df_train['date'].min().date()} — {df_train['date'].max().date()})")
print(f"Test : {len(df_test):5d} obs  "
      f"({df_test['date'].min().date()} — {df_test['date'].max().date()})")

# ── Scale features using train only (anti-leakage) ───────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(df_train[BASE_FEATURES].values)
X_test  = scaler.transform(df_test[BASE_FEATURES].values)
print(f"Scaled X_train: {X_train.shape}   X_test: {X_test.shape}")


In [ ]:
def compute_metrics(y_true, y_pred, y_naive, label=""):
    mse       = float(mean_squared_error(y_true, y_pred))
    rmse      = float(np.sqrt(mse))
    mae       = float(mean_absolute_error(y_true, y_pred))
    r2        = float(r2_score(y_true, y_pred))
    mse_naive = float(mean_squared_error(y_true, y_naive))
    mse_ratio = mse / mse_naive if mse_naive > 0 else float("nan")
    dir_acc   = float(np.mean(np.sign(y_pred) == np.sign(y_true)) * 100)
    return {"label": label, "MSE": mse, "RMSE": rmse, "MAE": mae,
            "R2": r2, "MSE_ratio": mse_ratio, "Dir_acc_pct": dir_acc}

def print_metrics_table(results_dict):
    header = f"  {'Model':<28} {'MSE':>10} {'RMSE':>10} {'MAE':>10} {'R2':>8} {'MSE/naive':>10} {'Dir%':>7}"
    print(header)
    print("  " + "-"*78)
    for model, m in results_dict.items():
        print(f"  {model:<28} {m['MSE']:>10.6f} {m['RMSE']:>10.6f} "
              f"{m['MAE']:>10.6f} {m['R2']:>8.4f} {m['MSE_ratio']:>10.4f} "
              f"{m['Dir_acc_pct']:>6.1f}%")


## Model 1: Naive Baseline

δ̂ = 0 for all parameters. Tomorrow's parameter = today's. This is the hardest-to-beat baseline for I(0) series with low autocorrelation in changes.

## Model 2: Gradient Boosting (conservative)

Conservative settings: 100 trees max, depth=3, lr=0.05, subsample=0.8, early stopping with validation fraction=0.15.


In [ ]:
results     = {}   # {param: {model: metrics}}
predictions = {}   # {param: {model: array}}

for p in params_in_data:
    tgt         = TARGETS[p]
    y_train     = df_train[tgt].values
    y_test      = df_test[tgt].values
    y_naive     = np.zeros_like(y_test)   # delta = 0 forecast
    results[p]     = {}
    predictions[p] = {}

    # ── Naive baseline ────────────────────────────────────────────────────────
    naive_mse = float(mean_squared_error(y_test, y_naive))
    results[p]["Naive"] = {
        "label": "Naive", "MSE": naive_mse, "RMSE": float(np.sqrt(naive_mse)),
        "MAE": float(mean_absolute_error(y_test, y_naive)),
        "R2": float(r2_score(y_test, y_naive)),
        "MSE_ratio": 1.0,
        "Dir_acc_pct": float(np.mean(np.sign(y_naive) == np.sign(y_test)) * 100)
    }
    predictions[p]["Naive"] = y_naive

    # ── Gradient Boosting ────────────────────────────────────────────────────
    gb = GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.05, max_depth=3,
        subsample=0.8, min_samples_leaf=10, random_state=42,
        validation_fraction=0.15, n_iter_no_change=15, tol=1e-4
    )
    gb.fit(X_train, y_train)
    p_gb = gb.predict(X_test)
    results[p]["GradientBoosting"] = compute_metrics(y_test, p_gb, y_naive, "GradientBoosting")
    predictions[p]["GradientBoosting"] = p_gb
    n_trees = getattr(gb, "n_estimators_", gb.n_estimators)
    print(f"  GB  {p:6s}: trees={n_trees:3d}  "
          f"R2={results[p]['GradientBoosting']['R2']:+.4f}  "
          f"MSE_ratio={results[p]['GradientBoosting']['MSE_ratio']:.4f}  "
          f"Dir={results[p]['GradientBoosting']['Dir_acc_pct']:.1f}%")

print("\nGradient Boosting done.")


## Model 3: Quantum Kernel SVR

### Methodology

**Feature map:** `ZZFeatureMap(feature_dimension=4, reps=1, entanglement="linear")`

This creates a 4-qubit circuit encoding 4 selected features via ZZ-interaction gates.
The circuit depth is kept minimal (reps=1) for interpretability and simulation speed.

**Kernel:** `FidelityQuantumKernel` — computes the Hilbert-space inner product
K(x,y) = |⟨ψ(x)|ψ(y)⟩|² between quantum feature states.

**SVR:** `SVR(kernel="precomputed", C=1.0, epsilon=0.01)` — trained on the quantum kernel matrix.

**Feature selection:** Correlation screening on train only — top 4 features by |corr| with target.
This is computed independently for each SSVI parameter target.

**Quantum training subset:** Last 150 observations of training set (temporal order preserved).
Runtime: ~150×150 = 22,500 kernel evaluations × 5 targets.

> ⚠️ If Qiskit is not installed, this section is skipped automatically.


In [ ]:
if QISKIT_AVAILABLE:
    from qiskit.circuit.library import ZZFeatureMap
    if QISKIT_KERNEL_API == "fidelity":
        from qiskit_machine_learning.kernels import FidelityQuantumKernel
    else:
        from qiskit_machine_learning.kernels import QuantumKernel

    N_QFEATURES  = 4
    Q_MAX_TRAIN  = 150   # kernel matrix: 150x150 = 22,500 evaluations per target
    q_feat_names = {}    # store selected feature names per target

    for p in params_in_data:
        tgt     = TARGETS[p]
        y_train = df_train[tgt].values
        y_test  = df_test[tgt].values
        y_naive = np.zeros_like(y_test)

        # Correlation screening (train only — anti-leakage)
        corrs   = np.array([abs(float(np.corrcoef(X_train[:, j], y_train)[0, 1]))
                             for j in range(X_train.shape[1])])
        top_idx = np.argsort(corrs)[::-1][:N_QFEATURES]
        q_feat_names[p] = [BASE_FEATURES[i] for i in top_idx]

        # Quantum subset: last Q_MAX_TRAIN train observations (temporal)
        X_q_tr  = X_train[-Q_MAX_TRAIN:][:, top_idx]
        y_q_tr  = y_train[-Q_MAX_TRAIN:]
        X_q_te  = X_test[:, top_idx]

        # Build quantum kernel
        fmap = ZZFeatureMap(feature_dimension=N_QFEATURES, reps=1, entanglement="linear")
        if QISKIT_KERNEL_API == "fidelity":
            qk = FidelityQuantumKernel(feature_map=fmap)
        else:
            qk = QuantumKernel(feature_map=fmap)

        print(f"  {p}: features={q_feat_names[p]}", flush=True)
        print(f"  {p}: computing K_train ({Q_MAX_TRAIN}x{Q_MAX_TRAIN})...", flush=True)
        K_train = qk.evaluate(x_vec=X_q_tr)

        print(f"  {p}: computing K_test  ({len(X_q_te)}x{Q_MAX_TRAIN})...", flush=True)
        K_test  = qk.evaluate(x_vec=X_q_te, y_vec=X_q_tr)

        svr = SVR(kernel="precomputed", C=1.0, epsilon=0.01)
        svr.fit(K_train, y_q_tr)
        p_qk = svr.predict(K_test)

        results[p]["QuantumKernelSVR"] = compute_metrics(
            y_test, p_qk, y_naive, "QuantumKernelSVR")
        predictions[p]["QuantumKernelSVR"] = p_qk

        m = results[p]["QuantumKernelSVR"]
        print(f"  {p}: R2={m['R2']:+.4f}  MSE_ratio={m['MSE_ratio']:.4f}  "
              f"Dir={m['Dir_acc_pct']:.1f}%\n")

    print("Quantum Kernel SVR done.")
else:
    print("Qiskit not available — quantum section skipped.")
    print("Install: pip install qiskit qiskit-machine-learning")


## Results Summary

MSE ratio < 1 = better than naive baseline. R² > 0 = beats naive. Directional accuracy > 50% = model predicts sign correctly more than chance.


In [ ]:
print("=" * 80)
print("SSVI PARAMETER DELTA FORECASTING — Full Results")
print("MSE ratio vs naive = 1.0. Lower is better. R2 > 0 beats naive.")
print("=" * 80)

for p in params_in_data:
    print(f"\n  Target: {TARGETS[p]}  (parameter: {p})")
    print_metrics_table(results[p])

# ── Compact comparison matrix ──────────────────────────────────────────────────
print("\n" + "=" * 80)
print("MSE RATIO MATRIX  (model_MSE / naive_MSE — lower is better)")
models = list(results[params_in_data[0]].keys())
header = f"  {'':12s}" + "".join(f"{m:>22s}" for m in models)
print(header)
for p in params_in_data:
    row = f"  {p:12s}"
    for m in models:
        mr = results[p].get(m, {}).get("MSE_ratio", float("nan"))
        tag = " <1.0" if mr < 1.0 else " >1.0"
        row += f"  {mr:>8.4f}{tag:5s}      "
    print(row)


## Visualizations


In [ ]:
# ── Plot 1: Actual vs Predicted delta per parameter ──────────────────────────
n_params = len(params_in_data)
fig, axes = plt.subplots(n_params, 1, figsize=(14, 3 * n_params), sharex=False)
if n_params == 1:
    axes = [axes]

for ax, p in zip(axes, params_in_data):
    y_test = df_test[TARGETS[p]].values
    dates  = df_test["date"].values
    ax.plot(dates, y_test, color="#333333", lw=0.8, label="Actual", alpha=0.7)
    colors = {"GradientBoosting": "#4477aa", "QuantumKernelSVR": "#cc3311"}
    for model, col in colors.items():
        if model in predictions[p]:
            ax.plot(dates, predictions[p][model], color=col, lw=0.9,
                    alpha=0.7, label=model)
    ax.axhline(0, color="gray", lw=0.5, ls="--")
    ax.set_title(f"d({p}) — actual vs predicted", fontsize=10)
    ax.set_ylabel(f"Δ{p}", fontsize=8)
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(True, alpha=0.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
fpath = PLOT_DIR / "qk_actual_vs_predicted.png"
plt.savefig(fpath, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {fpath}")


In [ ]:
# ── Plot 2: MSE ratio bar chart ───────────────────────────────────────────────
models_to_plot = [m for m in ["GradientBoosting", "QuantumKernelSVR"]
                  if m in results[params_in_data[0]]]
if models_to_plot:
    x = np.arange(len(params_in_data))
    w = 0.35
    fig, ax = plt.subplots(figsize=(10, 4))
    colors_bar = {"GradientBoosting": "#4477aa", "QuantumKernelSVR": "#cc3311"}
    for i, model in enumerate(models_to_plot):
        ratios = [results[p].get(model, {}).get("MSE_ratio", float("nan"))
                  for p in params_in_data]
        offset = (i - len(models_to_plot) / 2 + 0.5) * w
        bars = ax.bar(x + offset, ratios, w, label=model,
                      color=colors_bar.get(model, "#888888"),
                      edgecolor="white", alpha=0.85)
        for bar, r in zip(bars, ratios):
            if not np.isnan(r):
                ax.text(bar.get_x() + bar.get_width()/2, r + 0.005,
                        f"{r:.3f}", ha="center", va="bottom", fontsize=7)
    ax.axhline(1.0, color="black", lw=1.2, ls="--", label="Naive baseline (=1)")
    ax.set_xticks(x)
    ax.set_xticklabels(params_in_data)
    ax.set_ylabel("MSE ratio (vs naive)")
    ax.set_title("MSE ratio vs Naive baseline — lower is better", fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, axis="y", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    fpath = PLOT_DIR / "qk_mse_ratio.png"
    plt.savefig(fpath, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fpath}")


In [ ]:
# ── Plot 3: Scatter predicted vs actual (GB and QK-SVR) ──────────────────────
models_scatter = [m for m in ["GradientBoosting", "QuantumKernelSVR"]
                  if m in results[params_in_data[0]]]
if models_scatter:
    fig, axes = plt.subplots(len(params_in_data), len(models_scatter),
                              figsize=(5 * len(models_scatter), 3.5 * len(params_in_data)))
    if len(params_in_data) == 1:
        axes = [axes]
    if len(models_scatter) == 1:
        axes = [[ax] for ax in axes]

    for i, p in enumerate(params_in_data):
        y_test = df_test[TARGETS[p]].values
        for j, model in enumerate(models_scatter):
            ax = axes[i][j]
            if model not in predictions[p]:
                ax.axis("off")
                continue
            yp = predictions[p][model]
            ax.scatter(y_test, yp, alpha=0.3, s=8, color="#4477aa" if model == "GradientBoosting" else "#cc3311")
            lim = max(abs(y_test).max(), abs(yp).max()) * 1.1
            ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8, alpha=0.5)
            r2 = results[p][model]["R2"]
            ax.set_title(f"{model}  Δ{p}  R²={r2:.4f}", fontsize=8)
            ax.set_xlabel("Actual", fontsize=7)
            ax.set_ylabel("Predicted", fontsize=7)
            ax.grid(True, alpha=0.2)

    plt.tight_layout()
    fpath = PLOT_DIR / "qk_scatter.png"
    plt.savefig(fpath, dpi=110, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fpath}")


## SSVI Surface Reconstruction from Predicted Parameters

For three dates in the test set (normal day, high-error day, shock day), we reconstruct the
predicted SSVI implied-variance surface using:

$$w(k) = \frac{\theta_T}{2}\left[1 + \rho\gamma k + \sqrt{(\gamma k + \rho)^2 + (1 - \rho^2)}\right]$$

where $\theta_T = \eta \cdot T^\beta$ and $\gamma = \sqrt{2}\,\eta^{-1}\beta^{-1}$ (SSVI parametrization).

The predicted surface uses: `param_{t+1}^{pred} = param_t + Δ̂param_{t+1}` for each parameter.

> **Note:** Surface reconstruction requires all 5 parameters. If any target was not predicted,
> we fall back to carrying forward the current parameter (naive for that parameter).


In [ ]:
def ssvi_iv(k, alpha, beta, rho, eta, gamma, T=30/365):
    theta = eta * T**beta
    phi   = gamma  # simplified: using gamma directly as the phi parameter
    w     = (theta / 2) * (1 + rho * phi * k + np.sqrt((phi * k + rho)**2 + 1 - rho**2))
    return np.sqrt(np.maximum(w / T, 0))  # IV = sqrt(w/T)

def reconstruct_params(idx, model, predictions, df_test, params_in_data, TARGETS):
    row = df_test.iloc[idx]
    params = {}
    for p in params_in_data:
        current = row[p]
        if model in predictions.get(p, {}):
            pred_delta = predictions[p][model][idx]
        else:
            pred_delta = 0.0  # naive fallback for missing targets
        params[p] = current + pred_delta
    return params

# ── Select 3 reconstruction dates ─────────────────────────────────────────────
main_param = "alpha"
if main_param in results and "GradientBoosting" in results[main_param]:
    errors_gb = np.abs(predictions[main_param]["GradientBoosting"] -
                       df_test[TARGETS[main_param]].values)
    median_err = np.median(errors_gb)
    # Normal: closest to median error
    normal_idx = int(np.argmin(np.abs(errors_gb - median_err)))
    # High-error: 90th percentile
    high_err_idx = int(np.argmax(errors_gb))
    # Shock: highest VIX in test set (or highest RV change)
    if VIX_AVAILABLE and "VIX" in df_test.columns:
        shock_idx = int(df_test["VIX"].idxmax() % len(df_test))
    else:
        shock_idx = int(np.argmax(df_test["alpha"].diff().abs().fillna(0).values))

    recon_indices = [normal_idx, high_err_idx, shock_idx]
    recon_labels  = ["Normal day", "High-error day", "Shock/Stress day"]

    k_grid = np.linspace(-0.4, 0.3, 80)
    T30    = 30 / 365
    models_to_recon = [m for m in ["GradientBoosting", "QuantumKernelSVR"]
                       if m in results.get(main_param, {})]

    fig, axes = plt.subplots(len(recon_indices), 1, figsize=(10, 4 * len(recon_indices)))
    if len(recon_indices) == 1:
        axes = [axes]

    for ax, idx, label in zip(axes, recon_indices, recon_labels):
        row = df_test.iloc[idx]
        # Actual surface (next day's params)
        try:
            next_row = df_test.iloc[idx + 1]
            params_actual = {p: next_row[p] for p in params_in_data if p in next_row}
            iv_actual = ssvi_iv(k_grid, **{p: params_actual.get(p, row[p])
                                           for p in ["alpha","beta","rho","eta","gamma"]
                                           if p in params_in_data})
            ax.plot(k_grid, iv_actual, "k-", lw=2.0, label="Actual (t+1)", zorder=5)
        except Exception:
            pass

        colors_recon = {"GradientBoosting": "#4477aa", "QuantumKernelSVR": "#cc3311"}
        for model in models_to_recon:
            p_rec = reconstruct_params(idx, model, predictions, df_test, params_in_data, TARGETS)
            try:
                iv_pred = ssvi_iv(k_grid, **{p: p_rec.get(p, row[p])
                                              for p in ["alpha","beta","rho","eta","gamma"]
                                              if p in params_in_data})
                ax.plot(k_grid, iv_pred, color=colors_recon.get(model, "#888888"),
                        lw=1.5, ls="--", label=f"Predicted ({model})", alpha=0.85)
            except Exception as e:
                ax.text(0, 0.2, f"Reconstruction error: {e}", transform=ax.transAxes)

        date_str = str(row["date"])[:10] if "date" in row else f"idx={idx}"
        ax.set_title(f"{label} — {date_str}", fontsize=10)
        ax.set_xlabel("Log-moneyness k = log(K/F)")
        ax.set_ylabel("Implied Vol (annualized)")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.2)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.suptitle("SSVI Surface Reconstruction — Predicted vs Actual", fontsize=12, y=1.01)
    plt.tight_layout()
    fpath = PLOT_DIR / "qk_surface_reconstruction.png"
    plt.savefig(fpath, dpi=110, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fpath}")
else:
    print("Skipping surface reconstruction — GradientBoosting not run or alpha not available.")


## Research Findings

Auto-generated summary of empirical results.


In [ ]:
print("=" * 75)
print("RESEARCH FINDINGS — Quantum Kernel SVR for SSVI Parameter Forecasting")
print("=" * 75)

# ── 1. Which targets beat naive? ──────────────────────────────────────────────
print("\n[1] Models that beat the naive baseline (MSE_ratio < 1.0):")
for p in params_in_data:
    line = f"  {p:8s}: "
    for model in ["GradientBoosting", "QuantumKernelSVR"]:
        mr = results[p].get(model, {}).get("MSE_ratio", float("nan"))
        if not np.isnan(mr):
            tag = "BEATS naive" if mr < 1.0 else "WORSE than naive"
            line += f"{model}={mr:.4f} ({tag})  "
    print(line)

# ── 2. Quantum vs Gradient Boosting ──────────────────────────────────────────
if QISKIT_AVAILABLE and any("QuantumKernelSVR" in results[p] for p in params_in_data):
    print("\n[2] Quantum Kernel SVR vs Gradient Boosting (MSE ratio comparison):")
    qk_wins, gb_wins, ties = 0, 0, 0
    for p in params_in_data:
        r_gb = results[p].get("GradientBoosting", {}).get("MSE_ratio", float("nan"))
        r_qk = results[p].get("QuantumKernelSVR",  {}).get("MSE_ratio", float("nan"))
        if np.isnan(r_gb) or np.isnan(r_qk):
            continue
        if r_qk < r_gb - 0.005:
            outcome = "QK WINS"
            qk_wins += 1
        elif r_gb < r_qk - 0.005:
            outcome = "GB WINS"
            gb_wins += 1
        else:
            outcome = "TIE (~equal)"
            ties += 1
        print(f"  {p:8s}: GB={r_gb:.4f}  QK={r_qk:.4f}  -> {outcome}")
    print(f"  Summary: QK wins {qk_wins}/{len(params_in_data)}, "
          f"GB wins {gb_wins}/{len(params_in_data)}, ties {ties}")
else:
    print("\n[2] Quantum Kernel SVR: not run (Qiskit not available)")

# ── 3. Directional accuracy ───────────────────────────────────────────────────
print("\n[3] Directional accuracy (% correct sign prediction):")
for p in params_in_data:
    line = f"  {p:8s}: "
    for model in ["Naive", "GradientBoosting", "QuantumKernelSVR"]:
        da = results[p].get(model, {}).get("Dir_acc_pct", float("nan"))
        if not np.isnan(da):
            tag = "(>" + "50)" if da > 50 else "(<50)"
            line += f"{model}={da:.1f}%{tag}  "
    print(line)

# ── 4. Stress period performance ─────────────────────────────────────────────
print("\n[4] Performance in stress periods (test-set top-10% VIX days):")
if VIX_AVAILABLE and "VIX" in df_test.columns:
    vix_q90 = df_test["VIX"].quantile(0.90)
    stress_mask = df_test["VIX"] > vix_q90
    print(f"  Stress days: {stress_mask.sum()} (VIX > {vix_q90:.1f})")
    for p in params_in_data:
        y_full  = df_test[TARGETS[p]].values
        y_naive = np.zeros_like(y_full)
        print(f"  {p}:")
        for model in ["GradientBoosting", "QuantumKernelSVR"]:
            if model not in predictions[p]:
                continue
            yp       = predictions[p][model]
            y_stress = y_full[stress_mask]
            yp_stress = yp[stress_mask]
            yn_stress = y_naive[stress_mask]
            if len(y_stress) > 5:
                mse_s = float(mean_squared_error(y_stress, yp_stress))
                mse_n = float(mean_squared_error(y_stress, yn_stress))
                ratio_s = mse_s / mse_n if mse_n > 0 else float("nan")
                print(f"    {model}: stress_MSE_ratio={ratio_s:.4f}  "
                      f"(full_test_ratio={results[p][model]['MSE_ratio']:.4f})")
else:
    print("  VIX not available — skipping stress analysis")

# ── 5. Robustness warning ─────────────────────────────────────────────────────
print("\n[5] Robustness assessment:")
any_r2_positive = any(results[p].get("GradientBoosting", {}).get("R2", -99) > 0
                      for p in params_in_data)
if not any_r2_positive:
    print("  WARNING: No model achieves R2 > 0 on any target.")
    print("  SSVI parameter changes appear close to white noise at daily frequency.")
    print("  Naive baseline (delta=0) is competitive — this is expected for I(0) series.")
else:
    n_positive = sum(results[p].get("GradientBoosting", {}).get("R2", -99) > 0
                     for p in params_in_data)
    print(f"  {n_positive}/{len(params_in_data)} targets have R2 > 0 (GB).")

if QISKIT_AVAILABLE:
    qk_better = sum(
        results[p].get("QuantumKernelSVR", {}).get("MSE_ratio", 99) <
        results[p].get("GradientBoosting",  {}).get("MSE_ratio", 99) - 0.005
        for p in params_in_data)
    print(f"\n  Quantum kernel outperforms GB on {qk_better}/{len(params_in_data)} targets.")
    if qk_better == 0:
        print("  No quantum advantage detected in this experiment.")
        print("  The ZZFeatureMap kernel does not provide incremental predictive power")
        print("  over a conservative GradientBoosting on SSVI parameter changes.")
    else:
        print("  Marginal quantum advantage detected — interpret with caution.")
        print("  Result may be sensitive to quantum subset size and feature selection.")

print("\n" + "=" * 75)
print("End of Research Findings")
print("=" * 75)
